In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import HTML


fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")

ax.set_xlim(-6,6)
ax.set_ylim(-6,6)
ax.set_zlim(-6,6)
ax.set_box_aspect([1,1,1])
ax.grid(False)


# cube vertices
P = np.array([
[-1, 1, 1,-1,-1, 1, 1,-1],
[-1,-1, 1, 1,-1,-1, 1, 1],
[-1,-1,-1,-1, 1, 1, 1, 1]
])


faces = [
[0,1,2,3],
[4,5,6,7],
[0,1,5,4],
[1,2,6,5],
[2,3,7,6],
[3,0,4,7]
]


def Rx(t):
    return np.array([
        [1,0,0],
        [0,np.cos(t),-np.sin(t)],
        [0,np.sin(t), np.cos(t)]
    ])


def Rz(t):
    return np.array([
        [np.cos(t),-np.sin(t),0],
        [np.sin(t), np.cos(t),0],
        [0,0,1]
    ])


def draw_cube(P_transformed, color):

    face_polygons = []

    for f in faces:
        face = [[
            P_transformed[0,i],
            P_transformed[1,i],
            P_transformed[2,i]
        ] for i in f]

        face_polygons.append(face)

    cube = Poly3DCollection(
        face_polygons,
        facecolors=color,
        edgecolors="black",
        alpha=0.7
    )

    ax.add_collection3d(cube)



def update(frame):

    ax.clear()

    ax.set_xlim(-6,6)
    ax.set_ylim(-6,6)
    ax.set_zlim(-6,6)
    ax.set_box_aspect([1,1,1])
    ax.grid(False)


    theta = frame * 0.04

    R = Rz(theta) @ Rx(theta)


    # cube 1 (center rotating)
    P1 = R @ P
    draw_cube(P1, "cyan")


    # cube 2 (translating)
    tx = np.sin(theta) * 4
    T = np.array([tx,0,0])
    P2 = R @ P + T[:,None]

    draw_cube(P2, "orange")


    # cube 3 (orbit motion)
    tx = np.cos(theta) * 4
    ty = np.sin(theta) * 4
    T = np.array([tx,ty,0])

    P3 = R @ P + T[:,None]

    draw_cube(P3, "magenta")


    ax.set_title("Rotation + Translation (Multiple Cubes)")


ani = FuncAnimation(fig, update, frames=200, interval=50)

plt.close(fig)
HTML(ani.to_jshtml())

#Save the animation as an MP4 file
ani.save(
    "/home/shubh/projects/3d-anim/assets/gifs/Multiple_Cubes_translation_rotation.gif",
    writer="pillow",
    fps=60,
    dpi=200,
    bitrate=1800
)

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.set_xlim(-3,3)
ax.set_ylim(-3,3)
ax.set_zlim(0,6)

ax.set_box_aspect([1,1,1])
ax.grid(False)


# rotation around x axis (for limb swing)
def Rx(theta):

    return np.array([
        [1,0,0],
        [0,np.cos(theta),-np.sin(theta)],
        [0,np.sin(theta), np.cos(theta)]
    ])


# function to draw limb
def draw_limb(start, vec):

    end = start + vec

    ax.plot(
        [start[0],end[0]],
        [start[1],end[1]],
        [start[2],end[2]],
        color="black",
        linewidth=3
    )

    return end


def update(frame):

    ax.clear()

    ax.set_xlim(-3,3)
    ax.set_ylim(-3,3)
    ax.set_zlim(0,6)

    ax.set_box_aspect([1,1,1])
    ax.grid(False)


    t = frame * 0.08


    # body root
    hip = np.array([0,0,2])


    # torso
    torso_vec = np.array([0,0,2])
    neck = draw_limb(hip, torso_vec)


    # head
    ax.scatter(neck[0],neck[1],neck[2]+0.5,
               s=200,color="orange")


    # shoulder position
    shoulder = neck


    # arm swing
    arm_angle = np.sin(t)


    R_arm = Rx(arm_angle)

    upper_arm = R_arm @ np.array([0,0,-1.2])
    elbow = draw_limb(shoulder + np.array([0.4,0,0]), upper_arm)

    lower_arm = R_arm @ np.array([0,0,-1])
    draw_limb(elbow, lower_arm)


    R_arm2 = Rx(-arm_angle)

    upper_arm2 = R_arm2 @ np.array([0,0,-1.2])
    elbow2 = draw_limb(shoulder + np.array([-0.4,0,0]), upper_arm2)

    lower_arm2 = R_arm2 @ np.array([0,0,-1])
    draw_limb(elbow2, lower_arm2)


    # legs
    leg_angle = np.sin(t)


    R_leg = Rx(-leg_angle)

    upper_leg = R_leg @ np.array([0,0,-1.5])
    knee = draw_limb(hip + np.array([0.3,0,0]), upper_leg)

    lower_leg = R_leg @ np.array([0,0,-1.5])
    draw_limb(knee, lower_leg)


    R_leg2 = Rx(leg_angle)

    upper_leg2 = R_leg2 @ np.array([0,0,-1.5])
    knee2 = draw_limb(hip + np.array([-0.3,0,0]), upper_leg2)

    lower_leg2 = R_leg2 @ np.array([0,0,-1.5])
    draw_limb(knee2, lower_leg2)


    ax.set_title("Simple Human Motion (Articulated Figure)")


ani = FuncAnimation(fig, update, frames=200, interval=50)

plt.close(fig)
HTML(ani.to_jshtml())

ani.save(
    "/home/shubh/projects/3d-anim/assets/gifs/Human_Motion.gif",
    writer="pillow",
    fps=60,
    dpi=200,
    bitrate=1800
)
